# 04 · Ablations: training-data size and LoRA rank
Each run trains for 1 epoch on a fraction of the data and is evaluated on the fixed 300-question dev subset.
Runtime: **T4 GPU**, about 60–80 min in total. Run it after notebook 03, then re-run the last cell of 03 to add the table to the summary.

In [ ]:
# ---------- Setup: run this cell first ----------
import os, sys, subprocess, json, time, warnings
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less GPU memory fragmentation
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("T2S_SMOKE") == "1"      # local CPU smoke test (ignore on Colab/Kaggle)
ON_KAGGLE = os.path.exists("/kaggle/input")
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
from pathlib import Path
if ON_KAGGLE:
    # Code is attached as a Kaggle Dataset (read-only) -> copy it to the writable /kaggle/working
    import glob, shutil, zipfile
    PROJECT_DIR = "/kaggle/working/text2sql"

    def _walk(top="/kaggle/input"):
        return os.walk(top, followlinks=True)   # Kaggle may mount inputs via symlinks

    def _version(code_root):
        import re as _re
        m = _re.search(r"__version__\s*=\s*(\d+)", (Path(code_root) / "src/t2s/__init__.py").read_text())
        return int(m.group(1)) if m else 0

    def _find_code():
        zips, found = [], []
        for root, dirs, files in _walk():
            if root.endswith("src/t2s") and "modeling.py" in files and not root.startswith(PROJECT_DIR):
                found.append(Path(root).parents[1])
        if found:   # several attached versions -> use the newest code
            found.sort(key=_version, reverse=True)
            if len(found) > 1:
                print("several code copies found:", [(str(f), _version(f)) for f in found])
            return found[0]
        for root, dirs, files in _walk():
            zips += [Path(root) / f for f in files if f.lower().endswith(".zip") and "text2sql" in f.lower()]
        for z in zips:   # dataset kept as an un-extracted zip -> extract it ourselves
            tmp = Path("/kaggle/tmp/code"); tmp.mkdir(parents=True, exist_ok=True)
            zipfile.ZipFile(z).extractall(tmp)
            for root, dirs, files in os.walk(tmp):
                if root.endswith("src/t2s") and "modeling.py" in files:
                    return Path(root).parents[1]
        return None

    code_root = _find_code()
    if code_root is None:
        print("Could not find the text2sql code. Contents of /kaggle/input:")
        for root, dirs, files in _walk():
            depth = root.count(os.sep) - 2
            if depth <= 4:
                print("  " * depth + os.path.basename(root) + "/", (files[:5] if files else ""))
        raise RuntimeError("Code dataset not attached. Right panel -> Input -> '+ Add Input' -> Your Work -> Datasets -> "
                           "text2sql-code -> Add. Then run again. (Send the listing above if it is attached.)")
    print("code found at", code_root, "| code version", _version(code_root))
    # Kaggle images ship torchao 0.10, which makes peft refuse to load LoRA adapters on fp16 models. We don't need it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True)
    shutil.copytree(code_root, PROJECT_DIR, dirs_exist_ok=True,   # always refresh the code
                    ignore=shutil.ignore_patterns("artifacts", "results", "__pycache__"))
    for m in [m for m in list(sys.modules) if m == "t2s" or m.startswith("t2s.")]:
        del sys.modules[m]   # pick up new code if the kernel was not restarted
    # Resume support: if a previous run's Output is attached as an input, reuse its results/artifacts
    for root, dirs, files in _walk():
        if os.path.basename(root) == "results" and os.path.basename(os.path.dirname(root)) == "text2sql":
            prev_root = Path(root).parent
            for sub_dir in ["results", "artifacts", "reports"]:
                if (prev_root / sub_dir).exists():
                    shutil.copytree(prev_root / sub_dir, f"{PROJECT_DIR}/{sub_dir}", dirs_exist_ok=True)
            print("restored previous results from", prev_root)
    if not SMOKE:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"], check=True)
elif SMOKE:
    PROJECT_DIR = os.environ["T2S_PROJECT_DIR"]
elif not ON_COLAB:
    # Local machine (VS Code / Jupyter): the notebook lives in text2sql/notebooks/
    PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/masters-projects/text2sql"   # <- change if needed
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{PROJECT_DIR}/src")
ART, FIG, RES = f"{PROJECT_DIR}/artifacts", f"{PROJECT_DIR}/reports/figures", f"{PROJECT_DIR}/results"
for d in (ART, FIG, RES): os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
from t2s.spider import download_spider, find_spider_root, load_split, db_path, serialize_schema, schema_stats
from t2s import modeling as MD, evaluate as EV
from t2s.analysis import hardness, HARDNESS_ORDER

# ---- data ----
if SMOKE:
    from t2s.smoke import make_fake_spider
    SPIDER = find_spider_root(make_fake_spider(f"{PROJECT_DIR}/data"))
else:
    if ON_KAGGLE:
        attached = [os.path.join(r, "dev.json") for r, d, f in os.walk("/kaggle/input", followlinks=True)
                    if "dev.json" in f and os.path.isdir(os.path.join(r, "database"))]   # Spider attached as input?
        SPIDER = find_spider_root(Path(attached[0]).parent) if attached else download_spider("/kaggle/working/spider")
    elif ON_COLAB:
        SPIDER = download_spider("/content/spider", drive_zip="/content/drive/MyDrive/masters-projects/spider_data.zip")
    else:
        SPIDER = download_spider(f"{PROJECT_DIR}/data/spider", drive_zip=f"{PROJECT_DIR}/data/spider_data.zip")
train_all, dev = load_split(SPIDER, "train"), load_split(SPIDER, "dev")

# ---- model ----
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
if SMOKE:
    from t2s.smoke import make_tiny_model
    tiny = f"{ART}/tiny-model"
    if not os.path.exists(f"{tiny}/config.json"):
        make_tiny_model(tiny, [e.question + " " + e.query for e in train_all] + [serialize_schema(db_path(SPIDER, e.db_id), "ddl_rows") for e in train_all[:20] + dev])
    BASE_MODEL = tiny
DEV_SUBSET = sorted(np.random.RandomState(0).choice(len(dev), min(300, len(dev)), replace=False))  # fixed subset for ablations
SEED = 42
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()
def save_json(o, name):
    json.dump(o, open(f"{ART}/{name}.json", "w"), indent=2, default=float)
def load_json(name, default=None):
    p = f"{ART}/{name}.json"; return json.load(open(p)) if os.path.exists(p) else default
def cached(name, fn):
    """Run fn() once and store its DataFrame as results/<name>.parquet; later calls (after a crash) just load it."""
    p = f"{RES}/{name}.parquet"
    if os.path.exists(p):
        df = pd.read_parquet(p); print(f"[cached] {name}: EX={df.exec_match.mean():.4f}"); return df
    df = fn(); df.to_parquet(p, index=False); print(f"[done] {name}: EX={df.exec_match.mean():.4f}"); return df
def gen_cache(name):
    return f"{RES}/gen_{name}.jsonl"
print(f"train={len(train_all)}  dev={len(dev)}  model={BASE_MODEL}  device={MD.device_info()}")

In [ ]:
run = load_json("main_run"); BEST_FMT, MAX_LEN = run["schema_format"], run["max_len"]
tok = MD.load_tokenizer(BASE_MODEL)
rng = np.random.RandomState(SEED)
chats = EV.make_chats(train_all, SPIDER, BEST_FMT, with_answer=True)
all_rec, _ = MD.build_sft_records(tok, chats, MAX_LEN)
rng.shuffle(all_rec)
sub = [dev[i] for i in DEV_SUBSET]
CONFIGS = [  # (label, data fraction, rank) - all at 10% data: on a T4 a full epoch took ~7 h, so 10% ~ 40 min each
    ("10% data, r=16", 0.10, 16), ("10% data, r=4", 0.10, 4), ("10% data, r=64", 0.10, 64)]
if SMOKE: CONFIGS = CONFIGS[:2]

In [ ]:
import gc, torch
rows = []
for label, frac, r in CONFIGS:
    name = f"abl-f{int(frac*100)}-r{r}"
    if os.path.exists(f"{RES}/{name}.parquet"):
        df = pd.read_parquet(f"{RES}/{name}.parquet")
    else:
        recs = all_rec[: max(8, int(len(all_rec) * frac))]
        t0 = time.time()
        trainer = MD.train_lora(BASE_MODEL, recs, recs[:16], tok, f"{ART}/{name}", load_4bit=not SMOKE, r=r,
                                epochs=1, eval_steps=10**6, save_steps=50 if not SMOKE else 2, logging_steps=10,
                                max_steps=-1 if not SMOKE else 2, seed=SEED)
        trainer.model.save_pretrained(f"{ART}/adapter-{name}"); mins = (time.time() - t0) / 60
        del trainer; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
        m = MD.load_model(BASE_MODEL, load_4bit=False, adapter=f"{ART}/adapter-{name}", merge=True)
        df = EV.score(sub, MD.generate(m, tok, EV.make_chats(sub, SPIDER, BEST_FMT), batch_size=16), SPIDER)
        df["train_minutes"] = mins; df.to_parquet(f"{RES}/{name}.parquet", index=False)
        del m; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    rows.append({"config": label, "data_frac": frac, "rank": r, "exec_acc": df.exec_match.mean(),
                 "train_minutes": df.get("train_minutes", pd.Series([np.nan])).iloc[0]})
    print(rows[-1])

In [ ]:
ft = pd.read_parquet(f"{RES}/dev_finetuned.parquet").iloc[DEV_SUBSET]
zs = pd.read_parquet(f"{RES}/dev_zero-shot.parquet").iloc[DEV_SUBSET]
rows += [{"config": f"100% data, r={run['r']} (main)", "data_frac": 1.0, "rank": run["r"], "exec_acc": ft.exec_match.mean(),
          "train_minutes": run["train_minutes"]},
         {"config": "no fine-tuning (zero-shot)", "data_frac": 0.0, "rank": 0, "exec_acc": zs.exec_match.mean(), "train_minutes": 0}]
abl = pd.DataFrame(rows); abl.to_csv(f"{RES}/ablations.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
d = abl[(abl["rank"].isin([run["r"], 0]))].sort_values("data_frac")
ax[0].plot(d.data_frac * 100, d.exec_acc, "o-"); ax[0].set_xlabel("% of training data"); ax[0].set_ylabel("EX (subset)")
ax[0].set_title("Data scaling")
rk = abl[abl.data_frac == 0.10].sort_values("rank")
ax[1].plot(rk["rank"], rk.exec_acc, "o-"); ax[1].set_xscale("log", base=2); ax[1].set_xlabel("LoRA rank (10% data)")
ax[1].set_title("LoRA rank"); savefig("04_ablations"); abl.round(4)